# Attach institutional metadata and closure labels

Joins `processed_data/f2_model_ready.csv` to `raw_data/2024/hd2024.csv` to attach institution name, control, sector, state, and closure status. Builds a supervised target `closed_by_2024` for early-warning modeling.

**Closure heuristic (from hd2024 only):** an institution is labeled closed if any of:
- `CLOSEDAT` is set (not `-2`)
- `DEATHYR` is set (not `-2`)
- `CYACTIVE == 3` (currently inactive)

We also flag F2 panel members that do **not** appear in hd2024 at all (`in_hd2024 == False`) — these are likely pre-2024 closures that dropped off the directory.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
RAW = ROOT / "raw_data"
PROCESSED = ROOT / "processed_data"

model_ready = pd.read_csv(PROCESSED / "f2_model_ready.csv", low_memory=False)
model_ready.shape

In [ ]:
# hd2024 has a UTF-8 BOM on the first column header.
hd = pd.read_csv(RAW / "2024" / "hd2024.csv", encoding="utf-8-sig", low_memory=False)

HD_KEEP = ["UNITID", "INSTNM", "STABBR", "CONTROL", "SECTOR", "CLOSEDAT", "CYACTIVE", "DEATHYR", "ACT"]
hd = hd[HD_KEEP].copy()
hd.shape, hd.head(3)

In [ ]:
# IPEDS sentinel for "not applicable" is -2 (numeric) or '-2' (string in CLOSEDAT).
hd["closedat_set"] = hd["CLOSEDAT"].astype(str).str.strip() != "-2"
hd["deathyr_set"] = pd.to_numeric(hd["DEATHYR"], errors="coerce").fillna(-2) != -2
hd["inactive"] = pd.to_numeric(hd["CYACTIVE"], errors="coerce") == 3
hd["closed_by_2024"] = hd["closedat_set"] | hd["deathyr_set"] | hd["inactive"]

hd["closed_by_2024"].value_counts().to_dict()

In [ ]:
# Closure date as a parseable datetime (NaT when '-2').
hd["closed_date"] = pd.to_datetime(
    hd["CLOSEDAT"].where(hd["closedat_set"]),
    format="%m/%d/%Y", errors="coerce",
)
hd[["UNITID", "INSTNM", "closed_by_2024", "closed_date", "DEATHYR", "CYACTIVE"]][hd["closed_by_2024"]].head(10)

In [ ]:
# Left-join HD metadata onto the F2 panel.
hd_subset = hd[[
    "UNITID", "INSTNM", "STABBR", "CONTROL", "SECTOR", "ACT",
    "closed_by_2024", "closed_date",
]].copy()

labeled = model_ready.merge(hd_subset, on="UNITID", how="left", indicator=True)
labeled["in_hd2024"] = labeled["_merge"] == "both"
labeled = labeled.drop(columns=["_merge"])

# Institutions in the F2 panel but missing from hd2024 are likely older closures
# that dropped off the directory. Treat them as closed for the supervised label.
labeled["closed_by_2024"] = labeled["closed_by_2024"].fillna(True)

labeled.shape, labeled[["in_hd2024", "closed_by_2024"]].sum().to_dict()

In [ ]:
# Sanity check: label distribution at the institution (UNITID) level.
by_unit = labeled.groupby("UNITID").agg(
    in_hd2024=("in_hd2024", "max"),
    closed_by_2024=("closed_by_2024", "max"),
    n_years=("year", "nunique"),
    last_year=("year", "max"),
)
print("unique institutions:", len(by_unit))
print(by_unit[["in_hd2024", "closed_by_2024"]].sum().to_dict())
print("\nclosed-by-last-observed-year:")
print(by_unit[by_unit["closed_by_2024"]].groupby("last_year").size().to_string())

In [ ]:
out_path = PROCESSED / "f2_labeled.csv"
labeled.to_csv(out_path, index=False)
out_path, labeled.shape